In [1]:
import os
os.chdir(r'C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444')

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input, SimpleRNN
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
def create_rnn():
    input_data = Input(shape=(time_steps, num_features))
    rnn_layer1 = SimpleRNN(8, return_sequences=True)(input_data)
    rnn_layer2 = SimpleRNN(20)(rnn_layer1)
    x = Flatten()(rnn_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

In [5]:
model1 = create_rnn()
model1.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 simple_rnn (SimpleRNN)      (None, 24, 8)             240       
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, 20)                580       
                                                                 
 flatten (Flatten)           (None, 20)                0         
                                                                 
 dense (Dense)               (None, 1)                 21        
                                                                 
Total params: 841
Trainable params: 841
Non-trainable params: 0
_________________________________________________________________


In [6]:
import os

In [7]:
pip install pydot==1.2.4

Note: you may need to restart the kernel to use updated packages.


In [10]:
checkpoints = r'C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [11]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [12]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_rnn()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [14]:
import os
path_dataset =r'C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444'
path_tr = os.path.join(path_dataset, 'AEP_train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'AEP_test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

((84907, 21), (24259, 21), (12130, 21))

In [15]:
time_steps=24
num_features=21

In [16]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.6239058971405029 sec


In [17]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/60
2648/2653 [============================>.] - ETA: 0s - loss: 0.0339 - mae: 0.0339 - mape: 90.8776
Epoch 1: val_loss improved from inf to 0.01873, saving model to C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\E1-cp-0001-loss0.02.h5
2653/2653 [==============================] - 35s 12ms/step - loss: 0.0339 - mae: 0.0339 - mape: 90.7301 - val_loss: 0.0187 - val_mae: 0.0187 - val_mape: 9.3546
Epoch 2/60
2650/2653 [============================>.] - ETA: 0s - loss: 0.0143 - mae: 0.0143 - mape: 60.7385
Epoch 2: val_loss improved from 0.01873 to 0.01222, saving model to C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\E1-cp-0002-loss0.01.h5
2653/2653 [==============================] - 31s 12ms/step - loss: 0.0143 - mae: 0.0143 - mape: 60.6863 - val_loss: 0.0122 - val_mae: 0.0122 - val_mape: 6.2039
Epoch 3/60
2650/2653 [============================>.] - ETA: 0s - loss: 0.0131 - mae: 0.0131 - mape: 238.4206
Epoch 3: val_loss did not improve from 0.01222
2653/2653 [==============

2653/2653 [==============================] - 30s 11ms/step - loss: 0.0086 - mae: 0.0086 - mape: 271.5283 - val_loss: 0.0072 - val_mae: 0.0072 - val_mape: 3.1944
Epoch 24/60
2653/2653 [==============================] - ETA: 0s - loss: 0.0084 - mae: 0.0084 - mape: 281.8409
Epoch 24: val_loss did not improve from 0.00724
2653/2653 [==============================] - 32s 12ms/step - loss: 0.0084 - mae: 0.0084 - mape: 281.8409 - val_loss: 0.0086 - val_mae: 0.0086 - val_mape: 3.7140
Epoch 25/60
2651/2653 [============================>.] - ETA: 0s - loss: 0.0084 - mae: 0.0084 - mape: 250.5984
Epoch 25: val_loss did not improve from 0.00724
2653/2653 [==============================] - 31s 12ms/step - loss: 0.0084 - mae: 0.0084 - mape: 250.4522 - val_loss: 0.0079 - val_mae: 0.0079 - val_mape: 3.5473
Epoch 26/60
2650/2653 [============================>.] - ETA: 0s - loss: 0.0083 - mae: 0.0083 - mape: 335.9681
Epoch 26: val_loss did not improve from 0.00724
2653/2653 [=============================

Epoch 49/60
2652/2653 [============================>.] - ETA: 0s - loss: 0.0078 - mae: 0.0078 - mape: 99.6207
Epoch 49: val_loss did not improve from 0.00718
2653/2653 [==============================] - 34s 13ms/step - loss: 0.0078 - mae: 0.0078 - mape: 99.6000 - val_loss: 0.0073 - val_mae: 0.0073 - val_mape: 3.2688
Epoch 50/60
2649/2653 [============================>.] - ETA: 0s - loss: 0.0078 - mae: 0.0078 - mape: 40.7758
Epoch 50: val_loss did not improve from 0.00718
2653/2653 [==============================] - 32s 12ms/step - loss: 0.0078 - mae: 0.0078 - mape: 40.7245 - val_loss: 0.0086 - val_mae: 0.0086 - val_mape: 3.9587
Epoch 51/60
2653/2653 [==============================] - ETA: 0s - loss: 0.0077 - mae: 0.0077 - mape: 57.3111
Epoch 51: val_loss did not improve from 0.00718
2653/2653 [==============================] - 32s 12ms/step - loss: 0.0077 - mae: 0.0077 - mape: 57.3111 - val_loss: 0.0072 - val_mae: 0.0072 - val_mape: 3.1442
Epoch 52/60
2652/2653 [=======================

In [18]:

model = load_model(r'C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\E1-cp-0044-loss0.01.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 [==============================] - 2s 4ms/step
Mean Absolute Error (MAE): 115.43
Median Absolute Error (MedAE): 92.85
Mean Squared Error (MSE): 22861.78
Root Mean Squared Error (RMSE): 151.2
Mean Absolute Percentage Error (MAPE): 0.79 %
Median Absolute Percentage Error (MDAPE): 0.64 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


In [19]:
checkpoints = r'C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\E1-cp-0044-loss0.01.h5'
start_epoch= 54

In [20]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\E1-cp-0044-loss0.01.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [21]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
2649/2653 [============================>.] - ETA: 0s - loss: 0.0066 - mae: 0.0066 - mape: 34.7425
Epoch 1: val_loss improved from inf to 0.00674, saving model to C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\E2-cp-0001-loss0.01.h5
2653/2653 [==============================] - 35s 12ms/step - loss: 0.0066 - mae: 0.0066 - mape: 34.6986 - val_loss: 0.0067 - val_mae: 0.0067 - val_mape: 2.9145
Epoch 2/10
2648/2653 [============================>.] - ETA: 0s - loss: 0.0066 - mae: 0.0066 - mape: 35.4779
Epoch 2: val_loss did not improve from 0.00674
2653/2653 [==============================] - 30s 11ms/step - loss: 0.0066 - mae: 0.0066 - mape: 35.4205 - val_loss: 0.0068 - val_mae: 0.0068 - val_mape: 2.9324
Epoch 3/10
2653/2653 [==============================] - ETA: 0s - loss: 0.0066 - mae: 0.0066 - mape: 21.1768
Epoch 3: val_loss did not improve from 0.00674
2653/2653 [==============================] - 30s 11ms/step - loss: 0.0066 - mae: 0.0066 - mape: 21.1768 - val_loss: 0.006

In [22]:

model = load_model(r'C:\Users\Muhammad Luqman\Desktop\lab-9-21jzele0444\E2-cp-0010-loss0.01.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 [==============================] - 2s 4ms/step
Mean Absolute Error (MAE): 106.26
Median Absolute Error (MedAE): 83.32
Mean Squared Error (MSE): 19860.4
Root Mean Squared Error (RMSE): 140.93
Mean Absolute Percentage Error (MAPE): 0.73 %
Median Absolute Percentage Error (MDAPE): 0.58 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)
